# Эмбеддинги: Word2Vec и fastText — Лабораторная (стартовый шаблон)

**Версия:** стартовый ноутбук с разметкой клеток (теория / код / эксперименты).  
**Заполните все ячейки `TODO` и выполните их по порядку.**

**Сдача:** ноутбук `.ipynb` + краткий отчёт (2–4 стр.) по шаблону в конце.


## 0. Паспорт работы
- **Студент:** `впишите ФИО`
- **Группа:** `впишите`
- **Дата:** 2025-09-30
- **Тема:** Статические эмбеддинги слов: Word2Vec и fastText


In [ ]:
# TODO: укажите свои данные
STUDENT_NAME = ''
GROUP = ''
LAB_DATE = ''  # YYYY-MM-DD
print('OK — заполните поля выше перед началом работы.')

## 1. Подготовка окружения (не выполнять здесь, если запускаете локально с установленными пакетами)
Раскомментируйте и выполните, если работаете в Colab/свежем окружении.

> Примечание: интернет в среде может быть ограничен — используйте локально/Colab.


In [ ]:
# !pip install gensim==4.* fasttext==0.9.* numpy==1.* pandas==2.* scikit-learn==1.* umap-learn==0.*
# !pip install pymorphy2==0.9.* russian-tagsets==0.6.* # опционально для лемматизации

## 2. Теория (короткие ответы)
Ответьте кратко на вопросы ниже (5–8 предложений на вопрос).

**Q1.** В чём суть распределительной гипотезы и почему векторы слов работают?

**Q2.** Различия CBOW vs Skip-gram; negative sampling vs hierarchical softmax.

**Q3.** Почему fastText устойчив к OOV и морфологии? Роль символьных n-грамм.


## 3. Данные: подготовьте корпус
Выберите и подготовьте текстовый корпус на русском языке.

**Вариант A:** один текстовый файл `corpus.txt`, где документы разделены\nпереходами строк.  
**Вариант B:** папка `data/` с множеством `.txt` файлов, по документу/строке.

Сохраните в доступное место и укажите пути ниже. **Не выкладывайте приватные данные.**


In [ ]:
# TODO: пути к корпусу (ровно один из вариантов)
CORPUS_FILE = ''   # например: '/path/to/corpus.txt'
CORPUS_DIR = ''    # например: '/path/to/data_folder'

assert CORPUS_FILE or CORPUS_DIR, 'Укажите CORPUS_FILE или CORPUS_DIR'

### 3.1. Препроцессинг и токенизация
Определите токенизатор: нижний регистр, нормализация, опционально лемматизация.  
Убедитесь в консистентности (например, `ё→е`, единый дефис, латиница/кириллица).

In [ ]:
import re
from pathlib import Path

def simple_tokenize(text: str):
    # TODO: при необходимости замените на продвинутую токенизацию/лемматизацию
    text = text.lower()
    text = text.replace('ё', 'е')
    tokens = re.findall(r"[а-яa-z]+", text)
    return tokens

def read_sentences(corpus_file=None, corpus_dir=None, min_len=3):
    sents = []
    if corpus_file:
        # ожидается: одна строка ~ один документ; примитивно делим на предложения по пунктуации
        with open(corpus_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                for seg in re.split(r"[.!?]\s+", line):
                    toks = simple_tokenize(seg)
                    if len(toks) >= min_len:
                        sents.append(toks)
    elif corpus_dir:
        for p in Path(corpus_dir).glob('**/*.txt'):
            txt = p.read_text(encoding='utf-8', errors='ignore')
            for seg in re.split(r"[.!?]\s+", txt):
                toks = simple_tokenize(seg)
                if len(toks) >= min_len:
                    sents.append(toks)
    return sents

# TODO: выполните после указания путей
sentences = read_sentences(CORPUS_FILE or None, CORPUS_DIR or None)
print('Кол-во предложений:', len(sentences))
print('Пример:', sentences[0][:15] if sentences else '—')

## 4. Обучение Word2Vec
Настройте гиперпараметры и обучите модель. Экспериментируйте с `sg` (0=CBOW, 1=Skip-gram), `window`, `min_count`, `negative`, `sample`, `epochs`.

In [ ]:
from gensim.models import Word2Vec

# TODO: установите гиперпараметры
w2v_params = dict(
    vector_size=300,
    window=5,
    min_count=10,
    workers=4,
    sg=1,           # 1=Skip-gram, 0=CBOW
    negative=10,
    hs=0,
    sample=1e-5,
    epochs=10,
)

w2v = Word2Vec(sentences=sentences, **w2v_params)
print('Размер словаря:', len(w2v.wv))

### 4.1. Быстрые пробы (nearest neighbors, аналогии)

In [ ]:
# TODO: укажите свои термины
probe_words = ['данные', 'клиент', 'проект', 'рынок']
for q in probe_words:
    if q in w2v.wv:
        print(q, '→', [w for w,_ in w2v.wv.most_similar(q, topn=10)])
    else:
        print(q, '— нет в словаре')

def analogy(a,b,c, topn=5):
    try:
        return w2v.wv.most_similar(positive=[b,c], negative=[a], topn=topn)
    except KeyError:
        return 'Слова не в словаре'

print('Аналогия (пример):', analogy('мужчина','женщина','царь'))

## 5. Обучение fastText
Выберите один из вариантов: **native fasttext** или **gensim FastText**. Для морфологических языков (ru) используйте `minn=3, maxn=6`, `bucket≈2e6`.

In [ ]:
# Вариант A: native fasttext (работает с текстовым файлом)
# Подготовьте файл corpus.txt (токены через пробел, одна строка = один документ)
USE_FASTTEXT_NATIVE = False  # переключите в True, если хотите этот вариант
FASTTEXT_CORPUS_FILE = CORPUS_FILE  # при необходимости создайте отдельный файл

if USE_FASTTEXT_NATIVE:
    import fasttext
    assert FASTTEXT_CORPUS_FILE, 'Нужен путь к текстовому файлу корпуса (FASTTEXT_CORPUS_FILE)'
    ft = fasttext.train_unsupervised(
        input=FASTTEXT_CORPUS_FILE,
        model='skipgram',
        dim=300, ws=5, epoch=10, minCount=10,
        minn=3, maxn=6, bucket=2000000, lr=0.05
    )
    print('Готово: fastText (native)')
else:
    from gensim.models import FastText
    ft = FastText(
        sentences=sentences,
        vector_size=300, window=5, min_count=10,
        sg=1, minn=3, maxn=6, bucket=2000000, epochs=10
    )
    print('Готово: fastText (gensim)')

### 5.1. Пробы для fastText: соседи, OOV, опечатки

In [ ]:
test_words = ['закупщик', 'инженер', 'маркетолог']
for q in test_words:
    try:
        print('FT соседние для', q, '→', [w for w,_ in ft.wv.most_similar(q, topn=10)])
    except Exception as e:
        print(q, 'ошибка/нет вектора:', e)

# OOV / опечатки
for q in ['клиентоориентированность', 'маркетолг']:
    try:
        vec = ft.wv[q]  # gensim-сумма подслов, выдаст вектор даже если токена не было
        print(q, 'вектор OK, размер:', len(vec))
    except KeyError:
        try:
            # native fasttext вариант
            import fasttext
            if 'ft' in globals() and isinstance(ft, fasttext.FastText._FastText):
                vec = ft.get_word_vector(q)
                print(q, 'вектор OK (native)')
        except Exception as e:
            print(q, 'нет вектора:', e)

## 6. Intrinsic-оценка: сходство, аналогии, odd-one-out
Соберите мини-набор пар и аналогий для вашего домена. Посчитайте косинусы и Spearman ρ.

In [ ]:
import numpy as np
from scipy.stats import spearmanr  # при необходимости установите scipy

# TODO: заполните экспертные оценки (0..10)
pairs = [
    ('закупщик','байер', 9),
    ('выручка','маржа', 8),
    ('резюме','договор', 3),
    ('апельсин','сеньор', 0),
]

def cos(a, b, model):
    try:
        va, vb = model.wv[a], model.wv[b]
        return float(np.dot(va, vb) / (np.linalg.norm(va)*np.linalg.norm(vb)))
    except KeyError:
        return np.nan

def eval_pairs(model):
    gold = []
    sims = []
    rows = []
    for a,b,score in pairs:
        c = cos(a,b,model)
        gold.append(score)
        sims.append(c)
        rows.append((a,b,score,c))
    rho = spearmanr(gold, sims, nan_policy='omit').correlation
    return rho, rows

rho_w2v, rows_w2v = eval_pairs(w2v)
print('Spearman (W2V):', rho_w2v)
rho_ft, rows_ft = eval_pairs(ft)
print('Spearman (FT):', rho_ft)
rows_w2v, rows_ft[:2] if rows_ft else '—'

## 7. Extrinsic-оценка : классификация
Соберите небольшой датасет (текст → метка), постройте среднее эмбеддингов и обучите логистическую регрессию/линейный SVM.

In [ ]:
# TODO: подготовьте X_text, y (массива равной длины)
X_text = []  # список строк текстов
y = []       # список меток

def doc_embed(text, model):
    toks = simple_tokenize(text)
    vecs = [model.wv[t] for t in toks if t in model.wv]
    if not vecs:
        return np.zeros(model.wv.vector_size)
    return np.mean(vecs, axis=0)

if X_text:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score

    Xw = np.vstack([doc_embed(t, w2v) for t in X_text])
    Xf = np.vstack([doc_embed(t, ft) for t in X_text])

    Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, y, test_size=0.2, random_state=42)
    Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, y, test_size=0.2, random_state=42)

    clf_w = LogisticRegression(max_iter=200)
    clf_w.fit(Xw_tr, yw_tr)
    pred_w = clf_w.predict(Xw_te)
    print('F1 (W2V):', f1_score(yw_te, pred_w, average='macro'))

    clf_f = LogisticRegression(max_iter=200)
    clf_f.fit(Xf_tr, yf_tr)
    pred_f = clf_f.predict(Xf_te)
    print('F1 (FT):', f1_score(yf_te, pred_f, average='macro'))
else:
    print('Пропустите этот блок, если нет размеченного датасета.')

## 8. Визуализация: t-SNE / UMAP
Постройте 2D-проекции для 200–800 слов (без стоп-слов). Внимание к параметрам и нормализации.

In [ ]:
import numpy as np
# from sklearn.manifold import TSNE
# import umap
# import matplotlib.pyplot as plt

# TODO: сформируйте список слов subset
subset = list(w2v.wv.key_to_index.keys())[:500]
X = np.vstack([w2v.wv[w] for w in subset])
# Нормализация
X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)

# Пример t-SNE (раскомментируйте при наличии пакета)
# Z = TSNE(n_components=2, init='random', learning_rate='auto', perplexity=30, random_state=42).fit_transform(X)
# plt.figure(figsize=(8,8))
# plt.scatter(Z[:,0], Z[:,1], s=6)
# for i,w in enumerate(subset[:100]):
#     plt.text(Z[i,0], Z[i,1], w)
# plt.title('t-SNE для выборки слов (W2V)')
# plt.show()

## 9. Эксперименты: мини-грид гиперпараметров
Сравните ≥3 конфигурации и сведите результаты в таблицу.

In [ ]:
import itertools, pandas as pd
from gensim.models import Word2Vec

# TODO: задайте сетку
grid = {
    'sg': [0,1],            # CBOW/SG
    'window': [3,8],
    'vector_size': [200,300],
    'negative': [5,10],
}

results = []
for sg,window,vector_size,negative in itertools.product(grid['sg'],grid['window'],grid['vector_size'],grid['negative']):
    model = Word2Vec(
        sentences=sentences,
        sg=sg, window=window, vector_size=vector_size,
        min_count=10, negative=negative, epochs=5, workers=4
    )
    rho, _ = eval_pairs(model)
    results.append(dict(sg=sg, window=window, vector_size=vector_size, negative=negative, spearman=rho))
df = pd.DataFrame(results)
df.sort_values('spearman', ascending=False, inplace=True)
df

## 10. Итоговый отчёт (шаблон для копирования в документ)
- **Корпус:** источник, размер, примеры нормализации.
- **Гиперпараметры:** таблица конфигураций (W2V/FT) и краткие выводы.
- **Intrinsic:** 6–10 nearest neighbors + 6–10 аналогий, Spearman ρ.
- **Extrinsic (опц.):** F1/ROC-AUC или MRR@10.
- **Визуализация:** 1 изображение (t-SNE/UMAP) с пометками кластеров для FT и W2V.
- **Выводы:** где fastText выиграл/проиграл; что критично для домена; планы улучшения.

---
### Примечания
- Не выкладывайте приватные данные. Анонимизируйте примеры.
- Фиксируйте random seed там, где возможно, для повторяемости.
- Меняйте **по одному гиперпараметру за раз**, чтобы видеть причинно-следственную связь.
